# GRPO Multi-Agent Training

**Team SOYL — Round 2**

Train a Qwen2.5 policy against the `auction` / `dispute` / `coalition` tasks in the RF Spectrum multi-agent environment.

**Single-step formulation:** each training example is one observation; reward functions reset the env on that seed, step once, and read `metadata.reward_components`. This is what produces non-zero gradient signal with the current TRL release.

**Dual-backend:** flip `USE_UNSLOTH` in Cell 3 to `False` if Unsloth is being flaky. Vanilla transformers + PEFT + bitsandbytes is slower but doesn't have Unsloth's version-pinning fragility.

## Prereqs
- Colab secrets: `HF_TOKEN` (read/write), `WANDB_API_KEY`
- Runtime → Change runtime type → **T4 GPU**

## Cell 0 — GPU sanity check (RUN FIRST, EVERY TIME)

Stops the "Unsloth cannot find any torch accelerator" and "CUDA not available" errors at their source: if Colab didn't actually allocate a GPU, you find out here in 5 seconds instead of at the Unsloth import 3 minutes later.

In [ ]:
import subprocess, sys

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if smi.returncode != 0:
    print("nvidia-smi failed — Colab did NOT allocate a GPU.")
    print("Fix: Runtime -> Change runtime type -> T4 GPU, then Disconnect and delete runtime, then rerun.")
    print(smi.stderr)
    sys.exit(1)
print(smi.stdout.split("\n")[0:12][-1])

try:
    import torch
except ImportError:
    print("torch not installed yet — that's fine; Cell 1 installs it.")
else:
    print("torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device :", torch.cuda.get_device_name(0))
        print("CUDA memory :", f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    else:
        print("\nWARNING: nvidia-smi sees a GPU but torch.cuda does NOT.")
        print("Fix: !pip uninstall -y torch && pip install -q torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121")
        print("Then: Runtime -> Restart session, and rerun from Cell 0.")

## Cell 1 — Install dependencies

Installs both the Unsloth stack and the vanilla fallback stack in one go. If Unsloth's install fails for version-pin reasons, the vanilla stack still installs cleanly and Cell 3's `USE_UNSLOTH=False` branch works.

**After this cell: Runtime -> Restart session.** Colab caches imports from partial installs, which causes ~half of the "cannot import name" errors.

In [ ]:
%%capture
!pip install -q --upgrade pip

# Shared deps — needed by both backends.
!pip install -q trl peft accelerate bitsandbytes wandb openenv-core requests pydantic

# Unsloth's own Colab-new recipe (pinned versions they test against).
# If this line fails, the vanilla backend in Cell 6 will still work.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" || echo "Unsloth install skipped — use USE_UNSLOTH=False"
!pip install -q --no-deps "xformers<0.0.29"

# vllm only needed if you keep USE_UNSLOTH=True. Install conditionally with
# || true so a failure here doesn't block the vanilla path.
!pip install -q vllm || echo "vLLM install skipped — use USE_UNSLOTH=False"

## Cell 2 — Imports, secrets, W&B login

In [ ]:
import os, json, re, statistics, hashlib, gc

from google.colab import userdata
os.environ["HF_TOKEN"]       = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"]  = userdata.get("WANDB_API_KEY")

import requests
import torch
import wandb
wandb.login()

from trl import GRPOConfig, GRPOTrainer

## Cell 3 — Config: task, model, env, backend toggle

Flip `USE_UNSLOTH` to `False` if the Unsloth import fails or training hangs — the vanilla path below works identically, just ~2–3× slower per rollout.

In [ ]:
TASK = "auction"        # "auction" | "dispute" | "coalition"

SPACE_URL = "https://<org>-rf-spectrum-env-v2.hf.space"
# SPACE_URL = "http://127.0.0.1:7860"  # local docker fallback

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # fallback if 0.5B plateaus

USE_UNSLOTH = True      # flip to False if Unsloth misbehaves

TRAIN_SEEDS = list(range(200))          # 0..199
EVAL_SEEDS  = list(range(200, 230))     # held out

print(f"TASK={TASK}  MODEL_ID={MODEL_ID}  USE_UNSLOTH={USE_UNSLOTH}")

## Cell 4 — Thin HTTP wrapper over `server/app.py`

In [ ]:
class EnvHTTP:
    def __init__(self, base_url, timeout=30.0):
        self.base = base_url.rstrip("/")
        self.timeout = timeout

    def reset(self, seed=None, task_name=None, episode_index=None):
        payload = {}
        if seed is not None:          payload["seed"] = int(seed)
        if task_name is not None:     payload["task_name"] = task_name
        if episode_index is not None: payload["episode_index"] = int(episode_index)
        r = requests.post(f"{self.base}/reset", json=payload, timeout=self.timeout)
        r.raise_for_status()
        return r.json()

    def step(self, action):
        r = requests.post(f"{self.base}/step", json=action, timeout=self.timeout)
        r.raise_for_status()
        return r.json()

    def oversight(self):
        r = requests.get(f"{self.base}/oversight", timeout=self.timeout)
        r.raise_for_status()
        return r.json()

env = EnvHTTP(SPACE_URL)

## Cell 5 — Diagnostic: env round-trip and reward shape

If this cell raises, stop and fix the env. Training against a broken reward contract wastes hours.

In [ ]:
obs = env.reset(seed=0, task_name=TASK)
print("reset() keys:", sorted(obs.keys()))

trivial_action_by_task = {
    "auction":   {"bid_amount": 5.0, "justification": "test"},
    "dispute":   {"dispute_choice": "negotiate", "justification": "test"},
    "coalition": {"cooperation_flag": "cooperate", "justification": "test"},
}
obs2 = env.step(trivial_action_by_task[TASK])
meta = obs2.get("metadata") or {}

print("\ntop-level reward  :", obs2.get("reward"))
print("metadata keys     :", sorted(meta.keys()))
print("reward_components :", meta.get("reward_components"))

assert meta.get("reward_components"), (
    "env.step did not return metadata.reward_components — check server/spectrum_environment.py with Ryan."
)
print("\nOK: env round-trip works and returns reward_components.")

## Cell 6 — Load model (dual backend)

Branches on `USE_UNSLOTH`. Both paths end with `model` and `tokenizer` bound, and both populate a backend-agnostic `generate_text(prompt, temperature, max_tokens)` helper that the rest of the notebook uses. You never need to know which backend is active below this cell.

In [ ]:
if USE_UNSLOTH:
    try:
        from unsloth import FastLanguageModel
        from vllm import SamplingParams

        model, tokenizer = FastLanguageModel.from_pretrained(
            MODEL_ID,
            max_seq_length=2048,
            load_in_4bit=True,
            fast_inference=True,
            gpu_memory_utilization=0.5,
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=8,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_alpha=16,
        )

        def generate_text(prompt, temperature=0.0, max_tokens=256):
            out = model.fast_generate(
                [prompt],
                sampling_params=SamplingParams(
                    temperature=max(temperature, 1e-5),
                    max_tokens=max_tokens,
                ),
            )
            return out[0].outputs[0].text

        def switch_to_inference():
            FastLanguageModel.for_inference(model)

        def save_checkpoint(save_dir):
            model.save_pretrained_merged(save_dir, tokenizer, save_method="lora")

        def push_checkpoint(repo_id):
            model.push_to_hub_merged(
                repo_id, tokenizer, save_method="lora", token=os.environ["HF_TOKEN"]
            )

        def reload_from_disk(save_dir):
            global model, tokenizer
            model, tokenizer = FastLanguageModel.from_pretrained(
                save_dir,
                max_seq_length=2048,
                load_in_4bit=True,
                fast_inference=True,
                gpu_memory_utilization=0.5,
            )
            FastLanguageModel.for_inference(model)

        print("Loaded via Unsloth")

    except Exception as e:
        print(f"Unsloth load failed: {e!r}")
        print("Falling back to USE_UNSLOTH=False path. Set the flag in Cell 3 and rerun this cell.")
        raise

else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="eager",
    )
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=8, lora_alpha=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ))

    def generate_text(prompt, temperature=0.0, max_tokens=256):
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=temperature > 0,
                temperature=max(temperature, 1e-5),
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    def switch_to_inference():
        model.eval()

    def save_checkpoint(save_dir):
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)

    def push_checkpoint(repo_id):
        model.push_to_hub(repo_id, token=os.environ["HF_TOKEN"])
        tokenizer.push_to_hub(repo_id, token=os.environ["HF_TOKEN"])

    def reload_from_disk(save_dir):
        global model, tokenizer
        tokenizer = AutoTokenizer.from_pretrained(save_dir)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            attn_implementation="eager",
        )
        model = PeftModel.from_pretrained(base, save_dir)
        model.eval()

    print("Loaded via vanilla transformers + PEFT (USE_UNSLOTH=False)")

## Cell 7 — Build the single-step training dataset

In [ ]:
SYSTEM_PROMPT_BY_TASK = {
    "auction": (
        "You are a strategic bidder in a sealed-bid auction with two competitors. "
        "Your goal: maximise long-run reward while respecting your remaining budget. "
        "Lower bids preserve budget for future rounds; higher bids increase the chance "
        "of winning this round. Competitor bid history is revealed after each round. "
        'Respond with ONLY a JSON object: {"bid_amount": <float>, "justification": "<str>"}.'
    ),
    "dispute": (
        "You are a player in a one-shot dispute game. Pick one of "
        '{"concede","negotiate","escalate","audit"}. Payoff depends on your '
        "opponent's type, which you must infer from observable behavior. "
        'Respond with ONLY a JSON object: {"dispute_choice": "<one of the four>", "justification": "<str>"}.'
    ),
    "coalition": (
        "You are a player in an iterated coalition game with a reputation-tracking referee. "
        'Pick one of {"cooperate","defect","abstain"}. Cooperating raises your reputation; '
        "defecting lowers it and risks a regulator warning if your reputation is high. "
        'Respond with ONLY a JSON object: {"cooperation_flag": "<one of the three>", "justification": "<str>"}.'
    ),
}

def render_observation(obs):
    lines = [
        f"Round {obs.get('round_index', 0) + 1} of {obs.get('total_rounds', 1)}.",
        f"Your reputation: {obs.get('reputation_score', 0.5):.2f}.",
        f"Remaining budget: {obs.get('remaining_budget', 0.0):.2f}.",
        f"Competitor bid history: {obs.get('competitor_bid_history', [])}",
    ]
    events = obs.get("oversight_events", []) or []
    if events:
        lines.append("Recent oversight events:")
        for ev in events[-3:]:
            lines.append(
                f"  - {ev.get('event_type')} on {ev.get('operator_id')} "
                f"(sev {ev.get('severity', 0.0):.2f}): {ev.get('explanation', '')}"
            )
    cur = obs.get("current_request") or {}
    if cur.get("description"):
        lines.append(f"Scenario: {cur['description']}")
    return "\n".join(lines)

def build_chat_prompt(obs, task):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT_BY_TASK[task]},
        {"role": "user",   "content": render_observation(obs)},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

train_dataset = []
for seed in TRAIN_SEEDS:
    obs = env.reset(seed=seed, task_name=TASK)
    train_dataset.append({
        "prompt": build_chat_prompt(obs, TASK),
        "seed":   seed,
    })

print(f"Built {len(train_dataset)} single-step training prompts for task={TASK}")
print("\n--- Example prompt (seed 0) ---")
print(train_dataset[0]["prompt"][:1500])

## Cell 8 — Action parser + reward functions

Env access lives inside the reward functions; no `rollout_func` needed. The `_step_cache` collapses 4 reward-function calls per completion into 1 HTTP round-trip.

In [ ]:
def _extract_text(completion):
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list) and completion:
        first = completion[0]
        if isinstance(first, dict):
            return first.get("content", "")
        return str(first)
    return str(completion)

def _parse_action(text, task):
    text = re.sub(r"```(?:json)?", "", text).strip()
    data = {}
    try:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m: data = json.loads(m.group())
    except Exception:
        pass
    justification = str(data.get("justification", ""))[:500]

    if task == "auction":
        try:    bid = float(data.get("bid_amount", 0.0))
        except Exception: bid = 0.0
        return {"bid_amount": max(0.0, bid), "justification": justification}

    if task == "dispute":
        raw = str(data.get("dispute_choice", "negotiate")).lower().strip()
        if raw not in {"concede", "negotiate", "escalate", "audit"}:
            raw = "negotiate"
        return {"dispute_choice": raw, "justification": justification}

    raw = str(data.get("cooperation_flag", "cooperate")).lower().strip()
    if raw not in {"cooperate", "defect", "abstain"}:
        raw = "cooperate"
    return {"cooperation_flag": raw, "justification": justification}

_step_cache = {}

def _env_reward_components(seed, completion_text, task):
    key = (int(seed), completion_text)
    if key in _step_cache:
        return _step_cache[key]
    try:
        env.reset(seed=int(seed), task_name=task)
        action = _parse_action(completion_text, task)
        obs = env.step(action)
        meta = obs.get("metadata") or {}
        comps = meta.get("reward_components") or {}
        out = {
            "revenue":       float(comps.get("revenue",       meta.get("reward_revenue",       0.0))),
            "interference":  float(comps.get("interference",  meta.get("reward_interference",  0.0))),
            "compliance":    float(comps.get("compliance",    meta.get("reward_compliance",    0.0))),
            "justification": float(comps.get("justification", meta.get("reward_justification", 0.0))),
        }
    except Exception as e:
        print(f"[REWARD] env error on seed={seed}: {e}", flush=True)
        out = {"revenue": 0.0, "interference": 0.0, "compliance": 0.0, "justification": 0.0}
    if len(_step_cache) > 4096:
        _step_cache.clear()
    _step_cache[key] = out
    return out

def _make_reward_fn(component):
    def fn(completions, prompts=None, seed=None, **_):
        seeds = seed if seed is not None else [0] * len(completions)
        if isinstance(seeds, int):
            seeds = [seeds] * len(completions)
        scores = []
        for s, c in zip(seeds, completions):
            text = _extract_text(c)
            scores.append(_env_reward_components(s, text, TASK)[component])
        return scores
    fn.__name__ = f"extract_{component}"
    return fn

extract_revenue       = _make_reward_fn("revenue")
extract_interference  = _make_reward_fn("interference")
extract_compliance    = _make_reward_fn("compliance")
extract_justification = _make_reward_fn("justification")

# Sanity check before committing compute.
_dummy_completions_by_task = {
    "auction": [
        '{"bid_amount": 3.0, "justification": "Preserving remaining budget; competitor bid 7"}',
        '{"bid_amount": 99.0, "justification": "irrelevant"}',
    ],
    "dispute": [
        '{"dispute_choice": "negotiate", "justification": "Expected payoff favors negotiate given opponent mix"}',
        '{"dispute_choice": "escalate", "justification": "burn it down"}',
    ],
    "coalition": [
        '{"cooperation_flag": "cooperate", "justification": "Cooperating because reputation is below the 0.7 threshold"}',
        '{"cooperation_flag": "defect", "justification": "selfish"}',
    ],
}
dummy_completions = _dummy_completions_by_task[TASK]
dummy_seeds = [0, 1]
print("revenue:      ", extract_revenue(dummy_completions, seed=dummy_seeds))
print("interference: ", extract_interference(dummy_completions, seed=dummy_seeds))
print("compliance:   ", extract_compliance(dummy_completions, seed=dummy_seeds))
print("justification:", extract_justification(dummy_completions, seed=dummy_seeds))

If every number above is `0.0`, stop. The env contract is broken and training will be flat. Expected: at least `compliance` and `justification` return non-zero on the first dummy.

## Cell 9 — GRPOConfig (dual-backend aware)

`use_vllm` is only set on the Unsloth path — the vanilla backend uses standard HF generation inside the trainer.

In [ ]:
grpo_kwargs = dict(
    output_dir=f"rf-spectrum-{TASK}-lora",
    learning_rate=5e-6,
    num_generations=4 if USE_UNSLOTH else 2,    # vanilla is slower, cut generations
    max_prompt_length=1024,
    max_completion_length=256,
    num_train_epochs=1,
    max_steps=50,                               # smoke; raise to 150-300 for real run
    report_to="wandb",
    logging_steps=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    run_name=f"rf-spectrum-{TASK}-{'unsloth' if USE_UNSLOTH else 'vanilla'}-50step",
    remove_unused_columns=False,                # CRITICAL: keeps 'seed' in reward kwargs
)
if USE_UNSLOTH:
    grpo_kwargs["use_vllm"]   = True
    grpo_kwargs["vllm_mode"]  = "colocate"

config = GRPOConfig(**grpo_kwargs)

## Cell 10 — Train

In [ ]:
trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[
        extract_revenue,
        extract_interference,
        extract_compliance,
        extract_justification,
    ],
    args=config,
    train_dataset=train_dataset,
)
trainer.train()

### Pass criteria by step ~20

- `rewards/extract_*/std > 0` across all four components — non-zero variance = signal is flowing.
- `completions/clipped_ratio < 0.3` — completions terminate well under the 256-token cap.
- At least one component mean rising by step 50.

If any of those fail, cancel, revisit Cell 5 and Cell 8 sanity prints, and inspect raw completions from Cell 11 before burning more compute.

## Cell 11 — Sample generations on held-out seeds

In [ ]:
switch_to_inference()

def sample(seed, task=None, temperature=0.0, max_tokens=256):
    task = task or TASK
    obs = env.reset(seed=seed, task_name=task)
    prompt = build_chat_prompt(obs, task)
    text = generate_text(prompt, temperature=temperature, max_tokens=max_tokens)
    action = _parse_action(text, task)
    step_obs = env.step(action)
    meta = step_obs.get("metadata") or {}
    comps = meta.get("reward_components") or {}
    return {
        "seed": seed,
        "completion": text,
        "parsed_action": action,
        "reward_components": comps,
        "total_reward": step_obs.get("reward"),
    }

for s in EVAL_SEEDS[:5]:
    r = sample(s)
    print(f"\n--- seed {s} (held out) ---")
    print("completion:", r["completion"][:300])
    print("parsed action:", r["parsed_action"])
    print("reward components:", r["reward_components"])
    print("total reward:", r["total_reward"])

## Cell 12 — Save the checkpoint

In [ ]:
SAVE_DIR = f"rf-spectrum-{TASK}-trained"
save_checkpoint(SAVE_DIR)
print(f"Saved checkpoint to {SAVE_DIR} (backend={'unsloth' if USE_UNSLOTH else 'vanilla'})")

## Cell 13 — Post-save verification (mandatory)

Cold-reload the checkpoint to catch silent corruption. If the reloaded completions look like gibberish or the structure drifts from Cell 11's output, the save went bad.

In [ ]:
del trainer
gc.collect()
torch.cuda.empty_cache()

reload_from_disk(SAVE_DIR)

for s in EVAL_SEEDS[:5]:
    r = sample(s)
    print(f"\n[reloaded] seed {s}  reward={r['total_reward']}")
    print("completion:", r["completion"][:200])

## Cell 14 — Push to HF Hub

In [ ]:
ORG = "<org>"   # e.g. "team-soyl"
push_checkpoint(f"{ORG}/rf-spectrum-{TASK}-trained")
print(f"Pushed: https://huggingface.co/{ORG}/rf-spectrum-{TASK}-trained")

## Cell 15 — Held-out evaluation summary

In [ ]:
BASELINE_SCORES = {
    "auction":   0.25,
    "dispute":   0.30,
    "coalition": 0.20,
}

trained_scores = {}
for task in [TASK]:
    ep_rewards = []
    for s in EVAL_SEEDS:
        r = sample(s, task=task)
        ep_rewards.append(float(r["total_reward"] or 0.0))
    trained_scores[task] = round(statistics.mean(ep_rewards), 4)

print(f"{'Task':<16s}{'Baseline':>10s}{'Trained':>10s}{'Delta':>10s}")
print("-" * 46)
for task, trained in trained_scores.items():
    b = BASELINE_SCORES[task]
    delta = f"{((trained - b) / max(b, 0.01)) * 100:+.0f}%"
    print(f"{task:<16s}{b:>10.2f}{trained:>10.2f}{delta:>10s}")

---

## If training stalls or errors

| Symptom | Likely cause | Fix |
|:---|:---|:---|
| Cell 0: nvidia-smi fails | CPU-only runtime | Runtime -> Change runtime type -> T4 GPU, then Disconnect+delete runtime, rerun |
| Cell 0: `CUDA available: False` but nvidia-smi works | CPU-only torch got installed over CUDA torch | `!pip uninstall -y torch && pip install -q torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121`; restart session |
| Cell 6 (Unsloth path) raises any import error | Version pin conflict in the Unsloth stack | Set `USE_UNSLOTH = False` in Cell 3, rerun from Cell 6 |
| Cell 8 sanity prints all zeros | env `metadata.reward_components` missing | Ping Ryan; do not proceed |
| Cell 10: `rewards/*/std == 0` after 20 steps | `seed` not flowing through reward kwargs | Confirm `remove_unused_columns=False` in Cell 9 |
| Cell 10: `completions/clipped_ratio == 1.0` | Model isn't emitting stop tokens — chat template mismatch | Print `tokenizer.chat_template` and confirm it matches Qwen's default; restart session if it drifted |
| Cell 13: reloaded completions are gibberish | Save corrupted | Re-run Cell 12 with the other save path — for Unsloth swap `save_method="lora"` to `"merged_16bit"` |

## Running the other two tasks

1. Change `TASK` in Cell 3.
2. Rerun Cells 5, 7, 8, 9, 10, 11, 12, 13.
3. Cells 0, 1, 2, 4, 6 stay as-is — model is already loaded.